In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import shutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import os
import zipfile

import sys
sys.path.append('.')
from config import PROJECT_DIR, MAPS_ZIP, MAPS_DIR, LABELS, DATASET_ZIP, DATASET_DIR, MODELS_DIR

In [ ]:
os.system(f'unzip -q {DATASET_ZIP} -d {PROJECT_DIR}')

os.makedirs(f'{MAPS_DIR}', exist_ok=True)
print('.')

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
def apply_nclahe(image_np, tile_size):
    image_log = np.log1p(image_np.astype(np.float32))
    image_log = ((image_log - image_log.min()) /
                  (image_log.max() - image_log.min()) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(tile_size, tile_size))
    return clahe.apply(image_log)


class ChestXrayDataset(Dataset):
    def __init__(self, metadata_csv, split_csv, images_dir, resolution):
        self.df = pd.read_csv(metadata_csv)
        self.split_ids = pd.read_csv(split_csv)['image_id'].tolist()
        self.df = self.df[self.df['image_id'].isin(self.split_ids)]
        self.images_dir = images_dir
        self.resolution = resolution
        self.tile_size = 4 if resolution == 256 else 16
        self.image_ids = self.df['image_id'].unique().tolist()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img = cv2.imread(f'{self.images_dir}/{image_id}.png', cv2.IMREAD_GRAYSCALE)
        img = apply_nclahe(img, self.tile_size)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = Image.fromarray(img)
        img = transforms.ToTensor()(img)
        img = self.normalize(img)
        rows = self.df[self.df['image_id'] == image_id]
        label = torch.zeros(2)
        if 0 in rows['class_id'].values:
            label[0] = 1.0
        if 1 in rows['class_id'].values:
            label[1] = 1.0
        return img, label, image_id

In [ ]:
def build_model(architecture, resolution):
    if architecture == 'alexnet':
        model = models.alexnet(weights=None)
        model.classifier[6] = nn.Linear(4096, 2)
    elif architecture == 'densenet':
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(1024, 2)
    return model.to(DEVICE)


def get_target_layer(model, architecture):
    if architecture == 'alexnet':
        return model.features[12]
    elif architecture == 'densenet':
        return model.features.denseblock4

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach().clone()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach().clone()

    def generate(self, input_tensor, class_idx):
        input_tensor = input_tensor.requires_grad_(True)
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward(retain_graph=True)
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam.clone())
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

In [ ]:
# Verify resolution of saliency maps
for arq, res in [('alexnet', 256), ('alexnet', 1024), ('densenet', 256), ('densenet', 1024)]:
    if arq == 'alexnet':
        model = models.alexnet(weights=None)
        model.classifier[6] = nn.Linear(4096, 2)
        target = model.features[12]
    else:
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(1024, 2)
        target = model.features.norm5

    activation = {}
    def hook(module, input, output):
        activation['out'] = output

    target.register_forward_hook(hook)
    x = torch.zeros(1, 3, res, res)
    model(x)
    print(f"{arq} {res}x{res}: {activation['out'].shape}")

In [ ]:
configs = [
    {'name': 'AN_256',  'arq': 'alexnet',  'res': 256, 
     'metadata': f'{DATASET_DIR}/metadata_256.csv',
     'images':   f'{DATASET_DIR}/images_256'},
    {'name': 'DN_256',  'arq': 'densenet', 'res': 256,
     'metadata': f'{DATASET_DIR}/metadata_256.csv',
     'images':   f'{DATASET_DIR}/images_256'},
    {'name': 'AN_1024', 'arq': 'alexnet',  'res': 1024,
     'metadata': f'{DATASET_DIR}/metadata_1024.csv',
     'images':   f'{DATASET_DIR}/images_1024'},
    {'name': 'DN_1024', 'arq': 'densenet', 'res': 1024,
     'metadata': f'{DATASET_DIR}/metadata_1024.csv',
     'images':   f'{DATASET_DIR}/images_1024'},
]

for cfg in configs:
    name = cfg['name']
    print(f"\n{'='*50}")
    print(f"Generating maps for {name}...")

    out_dir = f'{MAPS_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    dataset = ChestXrayDataset(
        metadata_csv=cfg['metadata'],
        split_csv=f'{DATASET_DIR}/split_test.csv',
        images_dir=cfg['images'],
        resolution=cfg['res']
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    model = build_model(cfg['arq'], cfg['res'])
    pth = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth, map_location=DEVICE))
    model.eval()


    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        module._forward_pre_hooks.clear()

    target_layer = get_target_layer(model, cfg['arq'])
    gcam = GradCAM(model, target_layer=get_target_layer(model, cfg['arq']))

    for i, (img, label, image_id) in enumerate(loader):
        img = img.to(DEVICE)
        image_id = image_id[0]

        maps = {}
        for class_idx, label in enumerate(LABELS):
            cam =  gcam.generate(img, class_idx)
            res = cfg['res']
            maps[f'gradcam_{label}']   = cv2.resize(cam,   (res, res)).astype(np.float32)

        np.savez_compressed(f'{out_dir}/{image_id}.npz', **maps)

        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(dataset)} processed')

    print(f'  {name} completed.')

print('\nAll models completed.')

## Restore

In [ ]:
print('Compressing...')
with zipfile.ZipFile(f'{MAPS_ZIP}', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(f'{MAPS_DIR}'):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, f'{MAPS_DIR}')
            zf.write(filepath, arcname)

print('Done.')
print(f'Removing {DATASET_DIR} and {MAPS_DIR}...')
shutil.rmtree(DATASET_DIR)
shutil.rmtree(MAPS_DIR)

print('03_saliency_maps completed.')